# Erstellung der gocfl create Befehle

Dieses Jupyiter Notebook erstellt die gocfl create-Befehle für eine bestimmte Collection. 
Die Doku für den Aufbau eines gocfl create Befehls befindet sich hier: https://github.com/je4/gocfl/blob/main/docs/create.md

### Caveat: manuelles entzippen

Die AIPs liegen nun als ZIP-File im Ordner Objects, im Unterordner der jeweiligen Signatur. Es macht keinen Sinn, ZIP-Files in ein ZIP-Archiv zu ingesten. Daher müssen die ZIP-Files vor dem Ingest entzippt und die Ordner danach bereinigt werden (ZIP-Files löschen). Dies sind zum Zeitpunkt alles manuelle Prozesse. 

Dieses Notebook geht davon aus, dass im Unterordner objects/{signature}/ ein weiterer Unterordner liegt, dessen Content ins Archiv gelagert wird. Ist dies nicht der Fall, wird das Script abgebrochen.  
    

## config.py

In der Config wird die aktuell zu verarbeitende Collection sowie diverse Dateipfade konfiguriert. Bspw. für E-Manuscripta:

    collection_id = 'sosa_emanus'
    dlza_root = 'd:/Ingest'
    gocfl_conf = 'd:/Ingest/config/zhb-config.toml'

## signature

Die Signature ist zentral für die Erstellung des storage roots, sowie das Auffinden der Objekt-Pfade, Metadaten und Info-Dateien. Grundsätzlich sollte für jede Collection eine Textdatei namens '/signatures.txt' mit den signatures vorliegen. Diese werden entweder durch ein anderes Jupyter Notebook konfiguriert oder können von Hand erstellt werden.
Der Dateipfad kann konfiguriert werden. 

## objects
Das script prüft nicht, ob die SIP-Objekte tatsächlich vorhanden bzw. entzippt sind, dies muss vorher (manuell) überprüft werden. 


## storage root

Hier wird davon ausgegangen, dass der storage root ein ZIP file sein soll. Für jede signature wird ein storage_root angelegt. 


##  Create Befehl für gocfl generieren

Die gocfl create Befehle für alle Signaturen werden gemäss https://github.com/je4/gocfl/blob/main/docs/create.md erstellt. Der Pfad für die Config.toml kann konfiguriert werden.  

Muster:

    gocfl create ./archiv.zip ./object-directory metadata:./metadata-directory --config ./config/gocfl.toml -i 'signature'  --ext-NNNN-metafile-source ./info.json
    
Das Notebook erstellt für jedes Objekt eine Textdatei mit dem jeweiligen Create-Befehl.

## Display Befehl für gocfl generieren

Das Notebook erstellt zusätzlich den Display-Befehl gemäss https://github.com/je4/gocfl/blob/main/docs/display.md .
Der Display-Befehl wird in eine neue Zeile in das dazugehörige Create-Textfile geschrieben. 

Ausserdem wird der Name für die Report-Datei ins File geschrieben, so dass er auf der Workbench leicht kopiert werden kann.  

In [2]:
import config
import os
from zipfile import ZipFile
import shutil
from datetime import datetime
from pathlib import Path
import json

#prepare archive structure
root = config.dlza_root
coll = config.collection_id
org = config.organisation_id
files = config.gocfl_path
metadata =  f'{coll}/{config.metadata_path}'
info = f'{coll}/{config.info_path}'
objects = f'{coll}/{config.object_path}'
gocfl_conf = config.gocfl_conf
gocfl = config.gocfl

# filenames
f_create = f'{coll}/{files}/create_'
f_info = f'{config.inventory_file}'

# read line from file
with open(f_info, encoding="utf-8", errors='replace') as data_file:    
    data = json.load(data_file)
    for value in data:
        
        signature = value["signature"]
        foldername = value["references"][-1]
        print("foldername:",foldername)
        
        # create filepaths for metadata, info.json, objects:
        storage_root = f'{root}/{org}_{coll}_{foldername}.zip' 
        print("           Storage root: ",storage_root)
        dir_metadata = f'{root}/{metadata}/{foldername}/'
        print("           Metadata folder: ",dir_metadata)
        f_infojson = f'{root}/{info}/{foldername}.json'        
        print("           Info.json: ",f_infojson)
        dir_sip = f'{root}/{objects}/{foldername}/'
        print("           SIP folder: ",dir_sip)
        
        # create string  
        create_string = f'{gocfl} create {storage_root} {dir_sip} metadata:{dir_metadata} -i {signature} --ext-NNNN-metafile-source file://{f_infojson} --config {gocfl_conf}'
        print(f'\n###############\n{create_string}\n###############\n')
        
        # display string
        display_string = f'{gocfl} display {storage_root}'
        report_name = f'{org}_{coll}_{foldername}.pdf'
        
        # write strings to file
        create_file = f'{f_create}_{foldername}.txt'
        with open(create_file, 'w') as file:
            file.write(create_string)
            file.write('\n\n')
            file.write(display_string)
            file.write(f'\n\nSave report as: {report_name}')
               
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))                

foldername: 10_5076_e-codices-zhl-0034-4
           Storage root:  D:/Ingest/zhb_sosa_ecod_10_5076_e-codices-zhl-0034-4.zip
           Metadata folder:  D:/Ingest/sosa_ecod/metadata/10_5076_e-codices-zhl-0034-4/
           Info.json:  D:/Ingest/sosa_ecod/info/10_5076_e-codices-zhl-0034-4.json
           SIP folder:  D:/Ingest/sosa_ecod/objects/10_5076_e-codices-zhl-0034-4/

###############
gocfl create D:/Ingest/zhb_sosa_ecod_10_5076_e-codices-zhl-0034-4.zip D:/Ingest/sosa_ecod/objects/10_5076_e-codices-zhl-0034-4/ metadata:D:/Ingest/sosa_ecod/metadata/10_5076_e-codices-zhl-0034-4/ -i zhb:sosa_ecod_10_5076_e-codices-zhl-0034-4 --ext-NNNN-metafile-source file://D:/Ingest/sosa_ecod/info/10_5076_e-codices-zhl-0034-4.json --config D:/Ingest/config/zhb-config.toml
###############

foldername: 10_5076_e-codices-zhl-0040
           Storage root:  D:/Ingest/zhb_sosa_ecod_10_5076_e-codices-zhl-0040.zip
           Metadata folder:  D:/Ingest/sosa_ecod/metadata/10_5076_e-codices-zhl-0040/
       